In [ ]:
import sys
sys.path.insert(0, '../code')
import paths  # input and output locations, from config.yaml
import pandas as pd
import pylab as plt
import warnings
# Suppress warnings
warnings.filterwarnings('ignore')

In [ ]:
###########
fname = paths.der("population_by_macroregion_traveltime.csv")
df = pd.read_csv(fname)
###########
dfp = pd.read_csv(paths.der("population_by_macroregion.csv"))
dfp.columns = ['region', 'total_population']
df = df.merge(dfp, on='region')
df['rel_pop'] = 100*df['population'] / df['total_population']

REGION_TO_CONTINENT = {'Central Asia': 'Asia', 'Eastern Africa': 'Africa','Eastern Asia': 'Asia','Eastern Europe': 'Europe',
                        'Middle Africa': 'Africa', 'Northern Africa': 'Africa', 'Northern America': 'America',
                        'Northern Europe': 'Europe', 'Oceania': 'Oceania', 'South America': 'America', 'South-Eastern Asia': 'Asia',
                        'Southern Africa': 'Africa', 'Southern Asia': 'Asia', 'Southern Europe': 'Europe', 'Western Africa': 'Africa',
                        'Western Asia': 'Asia', 'Western Europe': 'Europe'}
df['continent'] = df['region'].map(REGION_TO_CONTINENT)
df

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
continents = df['continent'].unique()
palette = sns.color_palette("tab20", n_colors=20)

fig, axs = plt.subplots(3, 2, figsize=(10, 12), sharey=True)
axs = axs.flatten()

for i, continent in enumerate(continents):
    ax = axs[i]
    subset = df[df['continent'] == continent]
    subset = subset.sort_values(['rel_pop'])

    # Plot all bars at once with hue='region'
    sns.barplot(
        data=subset,
        x='threshold_min',
        y='rel_pop',
        hue='region',
        ax=ax,
        palette=palette,
        errorbar=None
    )
    ax.set_ylim([0,105])

fig.tight_layout()

In [ ]:
dfp = pd.read_csv(paths.der("population_by_macroregion.csv"))
dfp.columns = ['region', 'total_population']
dfp['continent'] = dfp['region'].map(REGION_TO_CONTINENT)
dfp = dfp.groupby('continent')['total_population'].sum().reset_index()
dfp

In [ ]:
dfTR = df.groupby(['continent', 'threshold_min'])['population'].sum().reset_index()
dfTR = dfTR.merge(dfp, on='continent')
dfTR['rel_pop'] = 100*dfTR['population'] / dfTR['total_population']

In [ ]:
from palettable.cartocolors.diverging import Geyser_5_r, Geyser_5
from palettable.cartocolors.qualitative import Safe_3, Antique_5, Vivid_7
from palettable.cubehelix import cubehelix3_16

dfTR = dfTR.sort_values('rel_pop')
CONTINENT_COLORS = {
    'Asia': Antique_5.hex_colors[0],
    'Africa': Safe_3.hex_colors[2],
    'Europe': cubehelix3_16.hex_colors[3],
    'America': Antique_5.hex_colors[3],
    'Oceania': Vivid_7.hex_colors[5]
}

f,ax=plt.subplots(figsize=(9, 5))
sns.barplot(x='threshold_min', y='rel_pop', hue='continent', ax=ax, data=dfTR, palette=CONTINENT_COLORS, alpha=0.75)

fname = paths.der("population_by_macroregion_traveltime.csv")
df = pd.read_csv(fname)
df_sum = df.groupby('threshold_min')['population'].sum().reset_index()
df_sum['pop_rel'] = 100*df_sum['population']/7830089931
df_sum['population'] = df_sum['population'] / 1e9
ax.plot(range(len(df_sum['threshold_min'])), df_sum['pop_rel'], color='black', lw=2, ls='--', label='Global population')


In [ ]:
dfTR.to_csv(paths.der("fig2b.csv"), index=None)